In [3]:
from datetime import timedelta
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import auth
from google.cloud import bigquery, storage

# Authenticate and set up environment
auth.authenticate_user()
project_id = 'mcxrp-429811'
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
!gcloud config set project {project_id}

# Initialize BigQuery client
client = bigquery.Client(project=project_id)

# Initialize GCP Storage client with user_project parameter
storage_client = storage.Client(project=project_id)
bucket_name = 'cxr_embedding'
bucket = storage_client.bucket(bucket_name, user_project=project_id)

# Path to the data in your GCP bucket
data_path = 'image-embeddings-mimic-cxr-1.0.physionet.org/files'

# Query BigQuery to get required columns
query = """
SELECT study.study_id, study.subject_id, study.study_datetime, study.path, record_list.dicom_id
FROM `physionet-data.mimic_cxr.study` AS study
JOIN `physionet-data.mimic_cxr.record_list` AS record_list
ON study.study_id = record_list.study_id
"""
query_job = client.query(query)
bigquery_df = query_job.to_dataframe()

# Load the ARDS cohort data table from GCP bucket
ards_data_path = 'ARDS cohort ~1000 patients 28 days July.csv'
ards_blob = bucket.blob(ards_data_path)
ards_blob.download_to_filename('/tmp/ards_data.csv')
ards_df = pd.read_csv('/tmp/ards_data.csv')

# Select required columns from ARDS cohort data
ards_filtered_df = ards_df[['subject_id', 'timezero', 'icu_mort', 'avg_peep', 'days_from_start']]

# Convert time columns to datetime format
bigquery_df['study_datetime'] = pd.to_datetime(bigquery_df['study_datetime'])
ards_filtered_df['timezero'] = pd.to_datetime(ards_filtered_df['timezero'])

# Keep only the rows where subject_id is in both dataframes
merged_subject_ids = set(bigquery_df['subject_id']).intersection(set(ards_filtered_df['subject_id']))
bigquery_df = bigquery_df[bigquery_df['subject_id'].isin(merged_subject_ids)]
ards_filtered_df = ards_filtered_df[ards_filtered_df['subject_id'].isin(merged_subject_ids)]



Updated property [core/project].


<ipython-input-3-53e0eeb9a4a9>:40: DtypeWarning: Columns (41) have mixed types. Specify dtype option on import or set low_memory=False.
  ards_df = pd.read_csv('/tmp/ards_data.csv')
<ipython-input-3-53e0eeb9a4a9>:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ards_filtered_df['timezero'] = pd.to_datetime(ards_filtered_df['timezero'])


In [ ]:
# Function to load embedding vectors from GCP bucket
def load_embedding_vector(file_path, dicom_id):
    clean_path = file_path.replace('.txt', '')
    if clean_path.startswith('files/'):
        clean_path = clean_path[len('files/'):]
    file_name = dicom_id + '.tfrecord'
    full_path = os.path.join(data_path, clean_path, file_name)
    blob = bucket.blob(full_path)

    #print(f"Checking if blob exists at path: {full_path}")  # Debugging log

    if blob.exists():
        #print(f"Blob found. Downloading data from: {full_path}")  # Debugging log
        try:
            raw_data = blob.download_as_bytes()
            print(f"Raw data size: {len(raw_data)} bytes")  # Debugging log

            # Save the downloaded data to a temporary file for TensorFlow to read
            temp_file_path = '/tmp/temp.tfrecord'
            with open(temp_file_path, 'wb') as f:
                f.write(raw_data)

            raw_dataset = tf.data.TFRecordDataset(temp_file_path)
            for raw_record in raw_dataset.take(1):
                example = tf.train.Example()
                example.ParseFromString(raw_record.numpy())
                embedding = example.features.feature['embedding'].float_list.value
                return embedding
        except tf.errors.DataLossError as e:
            print(f"Data loss error reading TFRecord file {full_path}: {e}")
        except tf.errors.InvalidArgumentError as e:
            print(f"Invalid argument error reading TFRecord file {full_path}: {e}")
        except Exception as e:
            print(f"General error reading TFRecord file {full_path}: {e}")
    else:
        print(f"File {full_path} does not exist")
    return None

# Create a new DataFrame to hold valid records
valid_rows = []

# Load embeddings into the DataFrame
for index, row in bigquery_df.iterrows():
    path = row['path']
    dicom_id = row['dicom_id']
    embedding = load_embedding_vector(path, dicom_id)
    if embedding:
        for i, val in enumerate(embedding):
            row[f'embedding_{i}'] = val
        valid_rows.append(row)

# Create a new DataFrame from valid rows
valid_df = pd.DataFrame(valid_rows)

# Merge the two dataframes on subject_id
merged_df = pd.merge(valid_df, ards_filtered_df, on='subject_id', how='inner')

# **Calculate the time difference between study_datetime and timezero**
merged_df['time_diff'] = merged_df['study_datetime'] - merged_df['timezero']

# **Filter out rows where the study_date_time is within 2 days before or within 1 day after the timezero**
filtered_df = merged_df[(merged_df['time_diff'] < -timedelta(days=2)) | (merged_df['time_diff'] > timedelta(days=1))]

# Sort by subject_id and time_diff, then drop duplicates keeping the closest study
#merged_df = merged_df.sort_values(by=['subject_id', 'time_diff']).drop_duplicates(subset='subject_id', keep='first')

# Reorder columns to have the embedding columns at the end
embedding_columns = [col for col in merged_df.columns if col.startswith('embedding_')]
non_embedding_columns = [col for col in merged_df.columns if not col.startswith('embedding_')]

final_columns = non_embedding_columns + embedding_columns
final_df = merged_df[final_columns]

# Save the final DataFrame to a CSV file in the GCP bucket
final_combined_data_path = 'final_combined_data1.csv'
final_combined_blob = bucket.blob(final_combined_data_path)
final_combined_blob.upload_from_string(final_df.to_csv(index=False), 'text/csv')
print(f"Final combined data saved to {final_combined_data_path} in GCP bucket")

Streaming output truncated to the last 5000 lines.
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
File image-embeddings-mimic-cxr-1.0.physionet.org/files/p12/p12691278/s51273308/9c081861-9bf48f61-7fb8881c-47299e0f-8ea2b5fe.tfrecord does not exist
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 bytes
Raw data size: 5698 

In [ ]:
auth.authenticate_user()
project_id = 'mcxrp-429811'  # Replace with your actual project ID
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id

# Initialize BigQuery client
client = bigquery.Client(project=project_id)

# Mount Google Drive
drive.mount('/content/drive')

# Load the extracted columns CSV file
extracted_columns_path = '/content/drive/My Drive/colab_data/extracted_columns.csv'
extracted_df = pd.read_csv(extracted_columns_path)

In [ ]:
dataset_id = 'mcxrp-429811.mimic_cxr1'  # Replace with your valid dataset ID
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"

In [ ]:
# Create the dataset
try:
    client.create_dataset(dataset)  # API request
    print(f"Created dataset {dataset_id}")
except Exception as e:
    print(f"Dataset {dataset_id} already exists or there was an error: {e}")

Created dataset mcxrp-429811.mimic_cxr1


In [ ]:
# Verify dataset and table names
dataset_id = 'mimic_cxr1'

# Query for CXR metadata from BigQuery
sql = f"""
CREATE VIEW `{dataset_id}.cxr_data` AS
SELECT * FROM `physionet-data.mimic_cxr.record_list`
"""
# Run the query and load the result into a DataFrame
try:
    client.query(sql).result()
    print(f"View cxr_data created successfully in dataset {dataset_id}")
except Exception as e:
    print(f"Error creating view: {e}")

View age1000 created successfully in dataset mimic_cxr1


In [ ]:
# Query the created view to load into a DataFrame
view_query = f"""
SELECT * FROM `{dataset_id}.cxr_data`
"""

# Run the query and load the result into a DataFrame
try:
    df_view = client.query(view_query).to_dataframe()
    # Display the resulting DataFrame
    display(df_view)
except Exception as e:
    print(f"Error querying view: {e}")

,subject_id,study_id,dicom_id,path
0,18415616,58605705,75d67482-46fbfcfb-b9d3be10-98f1b1dd-ba9748dc,files/p18/p18415616/s58605705/75d67482-46fbfcf...
1,18415616,58605705,91ea24c1-ddf8f918-0c579885-c0bf36ed-3a2b306a,files/p18/p18415616/s58605705/91ea24c1-ddf8f91...
2,19136768,50193341,69ea47d2-8e44c7ea-8fd5dada-9385460a-fd8863d2,files/p19/p19136768/s50193341/69ea47d2-8e44c7e...
3,19136768,58264435,c2822cd9-785880dd-b21df2f4-feae6873-a3dbcc34,files/p19/p19136768/s58264435/c2822cd9-785880d...
4,19136768,50193341,cbc8f5b5-f0eb7c26-5f128cde-62af96cb-9dd4f92a,files/p19/p19136768/s50193341/cbc8f5b5-f0eb7c2...
...,...,...,...,...
377105,17956863,54433918,766f17e3-6064ab91-59626504-e0570e87-d6e63c64,files/p17/p17956863/s54433918/766f17e3-6064ab9...
377106,17956863,54433918,a21b3a0b-fb10071c-0ab33ae4-0406b28b-a2e42bc3,files/p17/p17956863/s54433918/a21b3a0b-fb10071...
377107,17956863,58645749,4baead16-4c878e7c-fcc6071d-e06ebfb2-61d25c4c,files/p17/p17956863/s58645749/4baead16-4c878e7...
377108,17956863,50736896,328fba1d-fb7a2246-f361ecc5-09c96079-48a89f06,files/p17/p17956863/s50736896/328fba1d-fb7a224...


In [ ]:
if not extracted_df.empty and not df_view.empty:
    # Perform an inner join on 'subject_id' to get only matching records
    merged_df = pd.merge(extracted_df, df_view, on='subject_id', how='inner')
    # Display the resulting DataFrame
    display(merged_df)
else:
    print("One or both DataFrames are empty. Cannot perform merge.")

,subject_id,hadm_id,stay_id,timezero,icu_intime,icu_outtime,icu_mort,study_id,dicom_id,path
0,12595991,25205720,30027612,03/11/2147 17:38,03/11/2147 09:23,06/11/2147 19:25,1,50291999,09a7bc78-861b7d8a-bf31a633-67e32681-cec68e43,files/p12/p12595991/s50291999/09a7bc78-861b7d8...
1,12595991,25205720,30027612,03/11/2147 17:38,03/11/2147 09:23,06/11/2147 19:25,1,51474707,2fe309ca-e58c4d80-6f0002e9-cd535709-1c3f5890,files/p12/p12595991/s51474707/2fe309ca-e58c4d8...
2,12595991,25205720,30027612,03/11/2147 17:38,03/11/2147 09:23,06/11/2147 19:25,1,52076561,bd31fe67-ad4d5454-2cfd7c09-13c04383-d38297ac,files/p12/p12595991/s52076561/bd31fe67-ad4d545...
3,12595991,25205720,30027612,03/11/2147 17:38,03/11/2147 09:23,06/11/2147 19:25,1,55463602,bf9f8403-f941bbb9-13c134ff-ac80d6b9-e8442bdf,files/p12/p12595991/s55463602/bf9f8403-f941bbb...
4,12595991,25205720,30027612,03/11/2147 17:38,03/11/2147 09:23,06/11/2147 19:25,1,58585557,036272e9-9052e7c2-444e59fd-86a7f36d-9dfe191a,files/p12/p12595991/s58585557/036272e9-9052e7c...
...,...,...,...,...,...,...,...,...,...,...
11075,19046950,24352151,39998622,11/02/2135 22:15,11/02/2135 18:13,20/02/2135 17:53,0,57521020,ee407622-9577fed7-b1ddfa12-135f0f30-d9e9d6c1,files/p19/p19046950/s57521020/ee407622-9577fed...
11076,19046950,24352151,39998622,11/02/2135 22:15,11/02/2135 18:13,20/02/2135 17:53,0,59040853,da90fc30-f83bad51-67e557a8-86347b63-59b81b61,files/p19/p19046950/s59040853/da90fc30-f83bad5...
11077,19046950,24352151,39998622,11/02/2135 22:15,11/02/2135 18:13,20/02/2135 17:53,0,59568069,1668fc3c-a77bbd7a-83e11d91-10d68655-0466d48f,files/p19/p19046950/s59568069/1668fc3c-a77bbd7...
11078,19046950,24352151,39998622,11/02/2135 22:15,11/02/2135 18:13,20/02/2135 17:53,0,59642006,87094dfc-4dc02c35-932c18be-da0f9ee3-3b4ae3c1,files/p19/p19046950/s59642006/87094dfc-4dc02c3...


In [ ]:
# If you want to save the resulting DataFrame to a CSV file
output_path = '/content/drive/My Drive/colab_data/merged_patient_data.csv'
merged_df.to_csv(output_path, index=False)
print(f"Merged patient data saved to {output_path}")
############################################################################################################

Merged patient data saved to /content/drive/My Drive/colab_data/merged_patient_data.csv


In [ ]:
!pip install google-auth google-auth-oauthlib google-auth-httplib2 google-cloud-storage
